In [9]:
import os

In [10]:
file = 'data/day10.txt'
path = os.path.join(os.getcwd(), file)
with open(path, 'r') as fp:
    lines = [x.rstrip() for x in fp.readlines()]
data = [list(line) for line in lines]
grid = {}
for row, line in enumerate(data):
    for col, tile in enumerate(line):
        grid[(row, col)] = tile

In [11]:
def adjacent(tile, grid, directions='NSEW'):
    row, col, adjacent = *tile, []
    max_row = max([coord[0] for coord in grid]) 
    max_col = max([coord[1] for coord in grid])
    if row > 0 and 'N' in directions: # north
        adjacent.append((row - 1, col))
    if row < max_row and 'S' in directions: # south
        adjacent.append((row + 1, col))
    if col > 0 and 'W' in directions: # west
        adjacent.append((row, col - 1))
    if col < max_col and 'E' in directions: # east
        adjacent.append((row, col + 1))
    if row > 0 and col > 0 and 'nw' in directions: # northwest
        adjacent.append((row - 1, col - 1))
    if row > 0 and col < max_col and 'ne' in directions: # northeast
        adjacent.append((row - 1, col + 1))
    if row < max_row and col > 0 and 'sw' in directions: # southwest
        adjacent.append((row + 1, col - 1))
    if row < max_row and col < max_col and 'se' in directions: # southeast
        adjacent.append((row + 1, col + 1))
    return adjacent

In [12]:
start_tile = [i for i, x in grid.items() if x == 'S'][0]
pipe_connections = {'NS': '|', 'SN': '|', 'EW': '-', 'WE': '-',
                    'NE': 'L', 'EN': 'L', 'SE': 'F', 'ES': 'F',
                    'NW': '7', 'WN': '7', 'SW': 'J', 'WS': 'J'}
start_row, start_col = start_tile
start_pipe_connection, start_connections = '', []
north = adjacent(start_tile, grid, 'N') 
south = adjacent(start_tile, grid, 'S')
west = adjacent(start_tile, grid, 'W') 
east = adjacent(start_tile, grid, 'E')
if east and grid[east[0]] in ['J', '7', '-']:
    start_connections.append(east[0])
    start_pipe_connection += 'E'
if west and grid[west[0]] in ['L', 'F', '-']:
    start_connections.append(west[0])
    start_pipe_connection += 'W'
if south and grid[south[0]] in ['L', 'J', '|']:
    start_connections.append(south[0])
    start_pipe_connection += 'S'
if north and grid[north[0]] in ['7', 'F', '|']:
    start_connections.append(north[0])
    start_pipe_connection += 'N'
grid[start_tile] = pipe_connections[start_pipe_connection]

In [13]:
# part one
loop_dict = {start_tile: 0}
for coord in start_connections:
    count, curr_tile, prev_tile = 1, coord, start_tile
    while curr_tile != start_tile:
        row, col = curr_tile
        prev_row, prev_col = prev_tile
        curr_pipe = grid[curr_tile]
        match curr_pipe:
            case '|':
                row = row - 1 if prev_row > row else row + 1
            case '-':
                col = col - 1 if prev_col > col else col + 1
            case 'L':
                row = row - 1 if prev_col > col else row
                col = col + 1 if prev_row < row else col
            case 'J':
                row = row - 1 if prev_col < col else row
                col = col - 1 if prev_row < row else col
            case '7':
                row = row + 1 if prev_col < col else row
                col = col - 1 if prev_row > row else col
            case 'F':
                row = row + 1 if prev_col > col else row
                col = col + 1 if prev_row > row else col
        loop_dict[curr_tile] = min([loop_dict.get(curr_tile, count), count])
        prev_tile = curr_tile
        curr_tile = (row, col)
        count += 1
print(max(loop_dict.values()))

6778


In [14]:
# part two
right_list, left_list = set(), set()
curr_tile, prev_tile = start_connections[0], start_tile 
while True:
    row, col = curr_tile
    prev_row, prev_col = prev_tile
    curr_pipe = grid[curr_tile]
    n = adjacent(curr_tile, grid, 'N')
    s = adjacent(curr_tile, grid, 'S') 
    e = adjacent(curr_tile, grid, 'E')
    w = adjacent(curr_tile, grid, 'W')
    nw = adjacent(curr_tile, grid, 'nw')
    se = adjacent(curr_tile, grid, 'se')
    ne = adjacent(curr_tile, grid, 'ne')
    sw = adjacent(curr_tile, grid, 'sw')
    match curr_pipe:
        case '|':
            if prev_row > row:
                row -= 1
                if w and w[0] not in loop_dict:
                    left_list.add(w[0])
                if e and e[0] not in loop_dict:
                    right_list.add(e[0])
            else:
                row += 1
                if w and w[0] not in loop_dict:
                    right_list.add(w[0])
                if e and e[0] not in loop_dict:
                    left_list.add(e[0])
        case '-':
            if prev_col > col:
                col -= 1
                if n and n[0] not in loop_dict:
                    right_list.add(n[0])
                if s and s[0] not in loop_dict:
                    left_list.add(s[0])
            else:
                col += 1
                if n and n[0] not in loop_dict:
                    left_list.add(n[0])
                if s and s[0] not in loop_dict:
                    right_list.add(s[0])
        case 'L':
            if prev_row < row:
                col += 1
                if s and s[0] not in loop_dict:
                    right_list.add(s[0])
                if w and w[0] not in loop_dict:
                    right_list.add(w[0])
                if sw and sw[0] not in loop_dict:
                    right_list.add(sw[0])
                if ne and ne[0] not in loop_dict:
                    left_list.add(ne[0])
            else:
                row -= 1
                if s and s[0] not in loop_dict:
                    left_list.add(s[0])
                if w and w[0] not in loop_dict:
                    left_list.add(w[0])
                if sw and sw[0] not in loop_dict:
                    left_list.add(sw[0])
                if ne and ne[0] not in loop_dict:
                    right_list.add(ne[0])
        case 'F':
            if prev_row > row:
                col += 1
                if n and n[0] not in loop_dict:
                    left_list.add(n[0])
                if w and w[0] not in loop_dict:
                    left_list.add(w[0])
                if nw and nw[0] not in loop_dict:
                    left_list.add(nw[0])
                if se and se[0] not in loop_dict:
                    right_list.add(se[0])
            else:
                row += 1
                if n and n[0] not in loop_dict:
                    right_list.add(n[0])
                if w and w[0] not in loop_dict:
                    right_list.add(w[0])
                if nw and nw[0] not in loop_dict:
                    right_list.add(nw[0])
                if se and se[0] not in loop_dict:
                    left_list.add(se[0])
        case '7':
            
            if prev_row > row:
                col -= 1
                if n and n[0] not in loop_dict:
                    right_list.add(n[0])
                if e and e[0] not in loop_dict:
                    right_list.add(e[0])
                if ne and ne[0] not in loop_dict:
                    right_list.add(ne[0])
                if sw and sw[0] not in loop_dict:
                    left_list.add(sw[0])
            else:
                row += 1
                if n and n[0] not in loop_dict:
                    left_list.add(n[0])
                if e and e[0] not in loop_dict:
                    left_list.add(e[0])
                if ne and ne[0] not in loop_dict:
                    left_list.add(ne[0])
                if sw and sw[0] not in loop_dict:
                    right_list.add(sw[0])
        case 'J':
            if prev_row < row:
                col -= 1
                if s and s[0] not in loop_dict:
                    left_list.add(s[0])
                if e and e[0] not in loop_dict:
                    left_list.add(e[0])
                if se and se[0] not in loop_dict:
                    left_list.add(se[0])
                if nw and nw[0] not in loop_dict:
                    right_list.add(nw[0])
            else:
                row -= 1
                if s and s[0] not in loop_dict:
                    right_list.add(s[0])
                if e and e[0] not in loop_dict:
                    right_list.add(e[0])
                if se and se[0] not in loop_dict:
                    right_list.add(se[0])
                if nw and nw[0] not in loop_dict:
                    left_list.add(nw[0])
    if curr_tile == start_tile:
        break
    prev_tile = curr_tile
    curr_tile = row, col

In [15]:
in_out = {0: left_list, 1: right_list}
for i, old_list in enumerate([left_list, right_list]):
    while True:
        new_list = set()
        for coord in old_list:
            adj = adjacent(coord, grid, 'NSEWnenwsesw')
            adj = [x for x in adj if x not in loop_dict and x not in old_list]
            adj = [x for x in adj if x not in new_list and x not in in_out[i]]
            for coord in adj:
                new_list.add(coord)
        if not new_list:
            break
        in_out[i].update(new_list)
        old_list = new_list

In [16]:
if any(x[0] == 0 for x in in_out[0]) or any(x[1] == 0 for x in in_out[0]):
    inside = in_out[1]
    outside = in_out[0]
else:
    inside = in_out[0]
    outside = in_out[1]
total = set(grid.keys())
loop = set(loop_dict.keys())
print(len(inside))

433
